# Stock Market Predictor — Data Collection

Collecting 15 years of monthly data (2011–2026) from **Yahoo Finance** and **FRED**:

| Feature | Source | Ticker / Series |
|---|---|---|
| S&P 500 | Yahoo Finance | `^GSPC` |
| Crude Oil | Yahoo Finance | `CL=F` |
| US 10-Year Treasury Yield | Yahoo Finance | `^TNX` |
| Copper | Yahoo Finance | `HG=F` |
| Steel (PPI) | FRED | `WPU101707` |
| CPI (Inflation) | FRED | `CPIAUCSL` |
| Unemployment Rate | FRED | `UNRATE` |

## 1. Install Dependencies

We only need `yfinance` for Yahoo Finance data. FRED data is fetched directly via their public CSV endpoint, so no extra library is required.

In [7]:
%pip install yfinance -q

Note: you may need to restart the kernel to use updated packages.


## 2. Imports & Configuration

Define the 15-year window we want to pull data for. All downloads will use these shared start/end dates.

In [8]:
import yfinance as yf
import pandas as pd

START = "2011-01-01"
END = "2026-04-01"

## 3. Download Yahoo Finance Data

Pull **monthly closing prices** for four market indicators using `yfinance`. Each ticker is downloaded individually and the `Close` column is extracted into a single DataFrame.

In [9]:
yahoo_tickers = {
    "^GSPC": "SP500",
    "CL=F": "Crude_Oil",
    "^TNX": "US_10Y_Yield",
    "HG=F": "Copper",
}

yahoo_dfs = {}
for ticker, name in yahoo_tickers.items():
    df = yf.download(ticker, start=START, end=END, interval="1mo")
    if isinstance(df.columns, pd.MultiIndex):
        yahoo_dfs[name] = df[("Close", ticker)]
    else:
        yahoo_dfs[name] = df["Close"]

yahoo_df = pd.DataFrame(yahoo_dfs)
yahoo_df.index.name = "Date"
print(f"Yahoo Finance — {yahoo_df.shape[0]} rows, {yahoo_df.shape[1]} columns")
yahoo_df.tail()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Yahoo Finance — 184 rows, 4 columns


,SP500,Crude_Oil,US_10Y_Yield,Copper
Date,,,,
2025-12-01,6845.500000,57.419998,NaN,5.6300
2026-01-01,6939.029785,65.209999,NaN,5.8970
2026-02-01,6878.879883,NaN,NaN,NaN
2026-03-01,6368.850098,NaN,4.44,NaN
2026-03-27,NaN,101.180000,NaN,5.4615


## 4. Download FRED Data

Pull **monthly economic indicators** directly from the FRED public CSV endpoint — no API key needed. Each series is fetched individually and concatenated column-wise.

In [10]:
fred_series = {
    "WPU101707": "Steel_PPI",
    "CPIAUCSL": "CPI",
    "UNRATE": "Unemployment_Rate",
}

def fetch_fred(series_id, start, end):
    url = (
        f"https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={series_id}&cosd={start}&coed={end}"
    )
    df = pd.read_csv(url, parse_dates=["observation_date"], index_col="observation_date")
    df.columns = [series_id]
    return df

fred_df = pd.concat(
    [fetch_fred(sid, START, END) for sid in fred_series],
    axis=1,
)
fred_df.rename(columns=fred_series, inplace=True)
fred_df.index.name = "Date"
print(f"FRED — {fred_df.shape[0]} rows, {fred_df.shape[1]} columns")
fred_df.tail()

FRED — 182 rows, 3 columns


,Steel_PPI,CPI,Unemployment_Rate
Date,,,
2025-10-01,358.117,NaN,NaN
2025-11-01,367.621,325.063,4.5
2025-12-01,381.094,326.031,4.4
2026-01-01,395.358,326.588,4.3
2026-02-01,405.314,327.460,4.4


## 5. Combine into a Single DataFrame

Both datasets use slightly different date conventions, so we normalize each index to month-start timestamps before joining. An **inner join** keeps only months present in both sources, and rows with any missing values are dropped.

In [11]:
yahoo_df.index = yahoo_df.index.to_period("M").to_timestamp()
fred_df.index = fred_df.index.to_period("M").to_timestamp()

combined_df = yahoo_df.join(fred_df, how="inner")
combined_df.dropna(inplace=True)

print(f"Combined dataset: {combined_df.shape[0]} rows × {combined_df.shape[1]} columns")
print(f"Date range: {combined_df.index.min().strftime('%Y-%m')} → {combined_df.index.max().strftime('%Y-%m')}")
combined_df

Combined dataset: 138 rows × 7 columns
Date range: 2011-01 → 2024-05


,SP500,Crude_Oil,US_10Y_Yield,Copper,Steel_PPI,CPI,Unemployment_Rate
Date,,,,,,,
2011-01-01,1286.119995,92.190002,3.378,4.4510,220.400,221.187,9.1
2011-02-01,1327.219971,96.970001,3.414,4.4780,239.800,221.898,9.0
2011-03-01,1325.829956,106.720001,3.454,4.3000,248.100,223.046,9.0
2011-04-01,1363.609985,113.930000,3.296,4.1655,257.300,224.093,9.1
2011-06-01,1320.640015,95.419998,3.158,4.2720,250.300,224.806,9.1
...,...,...,...,...,...,...,...
2024-01-01,4845.649902,75.849998,3.967,3.9025,433.171,309.698,3.7
2024-02-01,5096.270020,78.260002,4.252,3.8345,454.090,310.967,3.9
2024-03-01,5254.350098,83.169998,4.206,4.0035,396.888,312.345,3.9


## 6. Export

Save the combined dataset to CSV.

In [12]:
combined_df.to_csv("data/combined_monthly.csv")
print("Saved to data/combined_monthly.csv")

Saved to data/combined_monthly.csv
